# 11.02 Claude Bash Tool Middleware

Claude 네이티브 `bash_20250124` 도구를 에이전트에 붙이는 `ClaudeBashToolMiddleware` 를 사용한다.
범용 `ShellToolMiddleware` 와 달리 Anthropic이 **서버에서 bash 호출 스키마를 직접 관리**하므로 프롬프트가 더 간결하고 tool schema 토큰이 절약된다.

## 학습 목표

- `ClaudeBashToolMiddleware` 의 4개 주요 파라미터(`workspace_root`, `startup_commands`, `execution_policy`, `redaction_rules`)를 이해한다
- 세 가지 실행 정책(`HostExecutionPolicy` / `DockerExecutionPolicy` / `CodexSandboxExecutionPolicy`)의 격리 수준과 트레이드오프를 구분한다
- `RedactionRule` 로 출력에서 민감정보(토큰, 키 등)를 마스킹한다
- Claude 네이티브 bash 도구가 범용 shell 도구와 무엇이 다른지 안다

## 언제 쓰나

- Claude에게 **실제 쉘 명령**을 실행시켜 코드 실행, 파일 조사, 빌드 같은 작업을 맡길 때
- Deep Agents처럼 장시간 작업 공간에서 **세션 상태**(cwd, 환경변수)를 유지해야 할 때
- **격리된 Docker 컨테이너**에서 안전하게 임의 코드를 실행하고 싶을 때
- 쉘 출력에 API 키·자격증명이 섞일 가능성이 있어 **출력 리다ek션**이 필요할 때

## 11.02.1 환경 설정

필요 패키지: `langchain`, `langchain-anthropic`. `.env`에 `ANTHROPIC_API_KEY` 가 있어야 한다.
Docker 정책 예시를 실행하려면 로컬에 Docker 데몬이 떠 있어야 한다.

In [ ]:
# !pip install -U langchain langchain-anthropic

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024)
model.invoke("ping").content[:50]

## 11.02.2 호스트 실행 정책 (가장 단순)

`HostExecutionPolicy` 는 **로컬 프로세스**에서 명령을 실행한다. 빠르고 설정이 없지만 **격리가 없다** —
네트워크, 파일시스템, 환경변수에 그대로 접근하므로 **신뢰할 수 있는 스크립트**나 로컬 개발 용도로만 쓴다.

- `workspace_root`: 쉘 세션의 기본 디렉터리
- `startup_commands`: 세션 시작 시 자동으로 실행되는 명령들 (PATH 설정, venv 활성화 등)

In [ ]:
import tempfile, pathlib
from langchain.agents import create_agent
from langchain.agents.middleware import HostExecutionPolicy
from langchain_anthropic.middleware import ClaudeBashToolMiddleware

workspace = pathlib.Path(tempfile.mkdtemp(prefix="bash-mw-"))
(workspace / "hello.txt").write_text("hi\n")

agent_host = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024),
    tools=[],
    middleware=[
        ClaudeBashToolMiddleware(
            workspace_root=str(workspace),
            startup_commands=["echo '세션 시작'", "pwd"],
            execution_policy=HostExecutionPolicy(),
        ),
    ],
)

result = agent_host.invoke({
    "messages": [{"role": "user", "content": "workspace 루트의 파일 목록을 출력하고 hello.txt 내용을 읽어줘."}]
})
print(result["messages"][-1].content[:400])

## 11.02.3 Docker 실행 정책 (권장, 격리)

`DockerExecutionPolicy` 는 명령을 **컨테이너 안**에서 실행한다. 호스트 파일시스템이나 네트워크에서 완전히 분리되므로,
**모델이 생성한 임의 코드**를 실행할 때 사실상 기본값으로 둬야 한다.

- `image`: 사용할 Docker 이미지 (기본 `python:3.11`)
- 컨테이너는 첫 bash 호출 시 생성되고 세션이 끝나면 정리된다
- 패키지 요구가 있다면 `startup_commands` 에 `pip install ...` 을 넣는다

In [ ]:
from langchain.agents.middleware import DockerExecutionPolicy

agent_docker = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024),
    tools=[],
    middleware=[
        ClaudeBashToolMiddleware(
            workspace_root="/workspace",  # 컨테이너 내부 경로
            startup_commands=[
                "mkdir -p /workspace",
                "cd /workspace",
                "pip install --quiet requests",
            ],
            execution_policy=DockerExecutionPolicy(image="python:3.11-slim"),
        ),
    ],
)

# Docker 데몬이 떠 있을 때만 실제로 돌려본다
# result = agent_docker.invoke({"messages": [{"role": "user", "content": "python -c 'import requests; print(requests.__version__)' 실행해줘"}]})
# print(result["messages"][-1].content[:400])
print("agent_docker ready (실제 실행은 Docker 데몬 필요)")

## 11.02.4 Codex 샌드박스 정책

`CodexSandboxExecutionPolicy` 는 Anthropic이 제공하는 **샌드박스 러너**에서 실행한다.
로컬 Docker 없이 격리를 얻고 싶을 때 대안이 된다. 네트워크 화이트리스트와 리소스 한도가 기본값으로 강하게 설정된다.

In [ ]:
from langchain.agents.middleware import CodexSandboxExecutionPolicy

agent_sandbox = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024),
    tools=[],
    middleware=[
        ClaudeBashToolMiddleware(
            workspace_root="/workspace",
            execution_policy=CodexSandboxExecutionPolicy(),
        ),
    ],
)
print("agent_sandbox ready")

## 11.02.5 출력 리다ek션 (`redaction_rules`)

쉘 출력에 API 키·토큰·이메일 같은 민감정보가 흘러나올 수 있다.
`RedactionRule(pattern=..., replacement=...)` 을 리스트로 넘겨 도구 응답을 **에이전트 컨텍스트에 담기 전에** 마스킹한다.
`pattern` 은 정규식 문자열, `replacement` 는 교체 문자열이다.

In [ ]:
from langchain.agents.middleware import RedactionRule

# langchain 1.2.x 의 RedactionRule 시그니처:
#   RedactionRule(pii_type, *, strategy, detector, apply_to_input, apply_to_output, apply_to_tool_results)
# 기존 (pattern=, replacement=) API 는 제거됨. detector 에 regex 문자열을 넘기면
# strategy="redact" 가 자동으로 `[REDACTED_<PII_TYPE>]` 형태로 치환한다.
redactions = [
    RedactionRule(
        pii_type="api_key",
        strategy="redact",
        detector=r"sk-[A-Za-z0-9]{20,}",
    ),
    RedactionRule(
        pii_type="email",
        strategy="redact",
        detector=r"[\w.+-]+@[\w-]+\.[\w.-]+",
    ),
]

agent_safe = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024),
    tools=[],
    middleware=[
        ClaudeBashToolMiddleware(
            workspace_root=str(workspace),
            execution_policy=HostExecutionPolicy(),
            redaction_rules=redactions,
        ),
    ],
)

# 모델이 `env` 등으로 출력한 민감정보가 [REDACTED_...] 로 치환된다
result = agent_safe.invoke({
    "messages": [{"role": "user", "content": "echo 'key=sk-ABCDEFGHIJKLMNOPQRSTUV user=alice@example.com' 를 실행해줘"}]
})
print(result["messages"][-1].content[:400])

## 11.02.6 Claude 네이티브 vs 범용 ShellToolMiddleware

| 항목 | `ClaudeBashToolMiddleware` | 범용 `ShellToolMiddleware` |
|------|---------------------------|----------------------------|
| 도구 타입 | Anthropic 네이티브 `bash_20250124` | 일반 `@tool` 함수 |
| 지원 모델 | Claude 전용 | 모든 모델 |
| tool schema 토큰 | 서버 측 → 0에 가까움 | 매 턴 전송 |
| 세션 상태 | 유지 (cwd, env 누적) | 구현에 따라 다름 |
| 실행 정책 API | 동일 (`HostExecutionPolicy` 등 공유) | 동일 |

**선택 기준**: Claude만 쓸 거면 네이티브가 더 싸고 깔끔하다. 멀티 프로바이더 파이프라인이라면 범용 `ShellToolMiddleware` 로 통일한다.

## 체크리스트

- [ ] 모델이 생성한 명령을 실행할 때 **`DockerExecutionPolicy` 또는 `CodexSandboxExecutionPolicy`** 를 기본으로 사용했는가
- [ ] `workspace_root` 가 대상 디렉터리 밖으로 탈출할 수 없는 경로인가
- [ ] 출력에 나올 수 있는 자격증명을 `redaction_rules` 로 커버했는가
- [ ] `startup_commands` 로 세션 초기화가 멱등한가 (여러 번 실행해도 안전한가)

## 다음 노트북

- `03_claude_text_editor.ipynb` — Claude 네이티브 텍스트 에디터 도구

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/middleware/anthropic
- Anthropic bash tool: https://docs.anthropic.com/claude/docs/tool-use-examples#bash-tool